# 🛍️ Shopping Mall Customer Segmentation
## Comparison — K-Means vs MeanShift vs DBSCAN
**Dataset:** Shopping_Mall_Customer_Segmentation_Data_.csv  
**Task:** Unsupervised Machine Learning — Full Comparison of Clustering Methods

## 1. Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans, MeanShift, DBSCAN, estimate_bandwidth
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score

import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully!')

## 2. Load Pre-processed Data
> Data was already cleaned and scaled in `step0_eda_preprocessing.ipynb`.


In [ ]:
# Load preprocessed data from step0_eda_preprocessing.ipynb output
# ⚠️ Make sure you have run step0_eda_preprocessing.ipynb first!
import numpy as np
import pandas as pd

X_scaled = np.load('X_scaled.npy')
df_clean  = pd.read_csv('data_preprocessed.csv')
df        = pd.read_csv('Shopping_Mall_Customer_Segmentation_Data_.csv')

print('✅ Loaded X_scaled.npy   shape:', X_scaled.shape)
print('✅ Loaded data_preprocessed.csv shape:', df_clean.shape)
print('✅ Loaded original CSV   shape:', df.shape)

In [ ]:
# ── K-Means ──────────────────────────────────────
# Find best K first
sil_scores = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbl = km.fit_predict(X_scaled)
    sil_scores.append(silhouette_score(X_scaled, lbl))
best_k = sil_scores.index(max(sil_scores)) + 2

kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
labels_km = kmeans.fit_predict(X_scaled)
print(f'K-Means: best K = {best_k}, clusters = {len(set(labels_km))}')

# ── MeanShift ─────────────────────────────────────
sample_size = min(3000, len(X_scaled))
np.random.seed(42)
sample_idx = np.random.choice(len(X_scaled), sample_size, replace=False)
bw = estimate_bandwidth(X_scaled[sample_idx], quantile=0.2, n_samples=500, random_state=42)
ms = MeanShift(bandwidth=bw, bin_seeding=True)
ms.fit(X_scaled)
labels_ms = ms.labels_
print(f'MeanShift: bandwidth={bw:.4f}, clusters = {len(set(labels_ms))}')

# ── DBSCAN ────────────────────────────────────────
# Quick param search
best_sil_db, best_eps, best_minsamp = -1, 0.5, 5
for eps in [0.3, 0.5, 0.7, 1.0, 1.2, 1.5]:
    for ms_val in [3, 5, 10]:
        db = DBSCAN(eps=eps, min_samples=ms_val)
        lbl = db.fit_predict(X_scaled)
        n_c = len(set(lbl)) - (1 if -1 in lbl else 0)
        if n_c >= 2:
            msk = lbl != -1
            s = silhouette_score(X_scaled[msk], lbl[msk]) if msk.sum() > 1 else -1
            if s > best_sil_db:
                best_sil_db, best_eps, best_minsamp = s, eps, ms_val

dbscan = DBSCAN(eps=best_eps, min_samples=best_minsamp)
labels_db = dbscan.fit_predict(X_scaled)
n_db = len(set(labels_db)) - (1 if -1 in labels_db else 0)
noise_db = (labels_db == -1).sum()
print(f'DBSCAN: eps={best_eps}, min_samples={best_minsamp}, clusters={n_db}, noise={noise_db}')

## 4. Side-by-Side Cluster Visualizations

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

method_labels = [('K-Means', labels_km), ('MeanShift', labels_ms), ('DBSCAN', labels_db)]

for ax, (name, labels) in zip(axes, method_labels):
    unique = sorted(set(labels))
    n_c = len(unique) - (1 if -1 in unique else 0)
    palette = sns.color_palette('tab10', n_c)
    color_map = {c: palette[i] for i, c in enumerate(l for l in unique if l != -1)}
    
    if -1 in unique:
        noise_msk = labels == -1
        ax.scatter(X_pca[noise_msk, 0], X_pca[noise_msk, 1],
                   c='lightgray', s=8, alpha=0.3, label='Noise', marker='x')
    
    for c in unique:
        if c == -1:
            continue
        msk = labels == c
        ax.scatter(X_pca[msk, 0], X_pca[msk, 1],
                   color=color_map[c], s=10, alpha=0.5, label=f'C{c}')
    
    noise_info = f' (+noise)' if -1 in unique else ''
    ax.set_title(f'{name}\n{n_c} clusters{noise_info}', fontsize=12, fontweight='bold')
    ax.set_xlabel('PCA 1')
    ax.set_ylabel('PCA 2')
    ax.legend(fontsize=6, loc='best', markerscale=1.5)

plt.suptitle('Clustering Comparison — PCA 2D View', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('comparison_pca.png', dpi=150)
plt.show()

## 5. Evaluation Metrics Comparison

In [ ]:
def evaluate(X, labels, name):
    mask = labels != -1
    n_c = len(set(labels)) - (1 if -1 in labels else 0)
    noise = (labels == -1).sum()
    if n_c >= 2 and mask.sum() > 1:
        sil = silhouette_score(X[mask], labels[mask])
        dbi = davies_bouldin_score(X[mask], labels[mask])
    else:
        sil, dbi = -1, -1
    inertia = KMeans(n_clusters=n_c, random_state=42, n_init=5).fit(X).inertia_ if n_c >= 2 else None
    return {'Method': name, 'Clusters': n_c, 'Noise Points': noise,
            'Silhouette ↑': round(sil, 4), 'Davies-Bouldin ↓': round(dbi, 4)}

results = [
    evaluate(X_scaled, labels_km, 'K-Means'),
    evaluate(X_scaled, labels_ms, 'MeanShift'),
    evaluate(X_scaled, labels_db, 'DBSCAN'),
]

results_df = pd.DataFrame(results).set_index('Method')
print('=' * 65)
print('         CLUSTERING METHODS EVALUATION COMPARISON')
print('=' * 65)
print(results_df.to_string())
print('=' * 65)
print('\n↑ = higher is better   ↓ = lower is better')
print('Silhouette: ranges -1 to 1, closer to 1 = well-separated clusters')
print('Davies-Bouldin: lower = more compact, well-separated clusters')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

methods = results_df.index.tolist()
colors = ['steelblue', 'coral', 'mediumseagreen']

axes[0].bar(methods, results_df['Silhouette ↑'], color=colors, edgecolor='white', width=0.5)
axes[0].set_title('Silhouette Score (higher = better)', fontweight='bold')
axes[0].set_ylabel('Silhouette Score')
axes[0].set_ylim(0, 1)
for i, v in enumerate(results_df['Silhouette ↑']):
    axes[0].text(i, v + 0.01, str(v), ha='center', fontsize=10, fontweight='bold')

axes[1].bar(methods, results_df['Davies-Bouldin ↓'], color=colors, edgecolor='white', width=0.5)
axes[1].set_title('Davies-Bouldin Index (lower = better)', fontweight='bold')
axes[1].set_ylabel('Davies-Bouldin Index')
for i, v in enumerate(results_df['Davies-Bouldin ↓']):
    axes[1].text(i, v + 0.01, str(v), ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Clustering Methods — Performance Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('comparison_metrics.png', dpi=150)
plt.show()

## 6. Summary Table & Discussion

In [ ]:
print('=' * 70)
print('              FINAL COMPARISON SUMMARY')
print('=' * 70)
print(results_df.to_string())
print('=' * 70)

best_sil_method = results_df['Silhouette ↑'].idxmax()
best_dbi_method = results_df['Davies-Bouldin ↓'].idxmin()

print(f'\n🏆 Best Silhouette Score : {best_sil_method} ({results_df.loc[best_sil_method, "Silhouette ↑"]})')
print(f'🏆 Best Davies-Bouldin   : {best_dbi_method} ({results_df.loc[best_dbi_method, "Davies-Bouldin ↓"]})')

print('\nMethod Characteristics:')
print('  K-Means   : Requires K upfront, fast, works well with spherical clusters')
print('  MeanShift : Auto-detects K, density-based, can be slow on large data')
print('  DBSCAN    : Auto-detects K, handles noise/outliers, finds arbitrary shapes')